# Notebook 01 — Dataset Analysis

This notebook explores the Excel files in the samples folder and establishes the structural patterns that will drive the parser design for the next notebooks.

In [10]:
from pathlib import Path
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'samples').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Could not find the repository root from the current notebook location.')

repo_root = find_repo_root(Path.cwd().resolve())
samples_dir = repo_root / 'samples'
excel_files = sorted(samples_dir.glob('*.xlsx'))

print('Repository root:', repo_root)
print('Samples directory:', samples_dir)
print('Excel files found:')
for path in excel_files:
    print('-', path.name)

Repository root: /home/yass/Desktop/DSS_CMR
Samples directory: /home/yass/Desktop/DSS_CMR/samples
Excel files found:
- Compo_All_Indices_20260731_copy.xlsx
- Data_sheet_without_legend.xlsx
- Données Marché Boursier_Projet_IA_copy.xlsx


In [11]:
def find_row_containing(df: pd.DataFrame, keywords: list[str]) -> int | None:
    for i in range(min(12, len(df))):
        row_text = ' '.join([str(x) for x in df.iloc[i].fillna('').tolist()]).lower()
        if all(keyword.lower() in row_text for keyword in keywords):
            return i
    return None

def detect_sheet_family(df: pd.DataFrame) -> str:
    head = df.iloc[:8, :].astype(str).fillna('')
    if head.apply(lambda col: col.str.contains('code isin', case=False).any()).any():
        return 'Family A'
    if head.iloc[:, 0].astype(str).str.strip().str.match(r'^[A-Z0-9\- /]+$').sum() >= 3:
        return 'Family B'
    if 'séance' in ' '.join(head.iloc[0, :].tolist()).lower() or 'seance' in ' '.join(head.iloc[0, :].tolist()).lower():
        return 'Index composition / normalized'
    return 'Unknown'

for workbook_path in excel_files:
    print(f'\n=== {workbook_path.name} ===')
    xls = pd.ExcelFile(workbook_path)
    print('Sheets:', xls.sheet_names)
    workbook_sheets = pd.read_excel(workbook_path, sheet_name=None)
    for sheet_name in xls.sheet_names:
        df = workbook_sheets[sheet_name]
        family = detect_sheet_family(df)
        metadata_hits = []
        for keywords in [['code isin'], ['libelle'], ['identifier', 'sub-libelle'], ['ask'], ['bid'], ['cours']]:
            row_idx = find_row_containing(df, keywords)
            if row_idx is not None:
                metadata_hits.append((keywords[0], row_idx))
        print(f'\n-- {sheet_name} -- family={family} shape={df.shape}')
        print('Metadata rows:', metadata_hits)
        print(df.head(6).to_string(index=False))
        print('---')


=== Compo_All_Indices_20260731_copy.xlsx ===
Sheets: ['MASI', 'Sector Indices', 'MASI 20', 'MASI ESG', 'MASI Mid and Small Cap']

-- MASI -- family=Family B shape=(19, 10)
Metadata rows: []
    Séance Indice    Code ISIN           Instrument  Cours  Nombre de titres  Facteur flottant  Facteur de plafonnement  Capitalisation flottante    Poids
2026-07-31   MASI MA0000012445    ATTIJARIWAFA BANK  685.0         215140839              0.30                        1              4.421144e+10 0.165426
2026-07-31   MASI MA0000012866              MANAGEM 1319.0         118646760              0.15                        1              2.347426e+10 0.087833
2026-07-31   MASI MA0000012312    SODEP-Marsa Maroc  860.0          73395600              0.35                        1              2.209208e+10 0.082662
2026-07-31   MASI MA0000011488 ITISSALAT AL-MAGHRIB   92.5         879095340              0.20                        1              1.626326e+10 0.060852
2026-07-31   MASI MA0000012320    

## Interpretation

- The market workbook is expected to contain sheets that follow a metadata-plus-columns layout, which is typical of Family A.
- The sheet with many repeated attribute names such as ALTHIGHMID, ALTLOWMID, VWAP, HVOLA, and 52W-HIGH is a strong candidate for Family B because it is organized as a block of variables rather than a simple column-per-company table.
- The index-composition workbook is already in a row-per-instrument format and should be treated as an already normalized dataset, not as a parser target for the Family A/B logic.

## Parser design preparation

1. Family A parser: detect metadata rows, locate CODE ISIN and company-name columns, identify the trading-date rows, and reshape the sheet into a long table keyed by Date and CODE ISIN.
2. Family B parser: detect the block structure, identify company blocks, map each variable row to the correct company, and produce the same normalized output shape.
3. Index-composition dataset: keep it separate, validate it, and join it later to the normalized market dataset using CODE ISIN.